In [ ]:
import requests
from IPython.display import display

base_url = "https://moustaphalabban.com/products.json"
all_products_data = []
page = 1
limit = 100

while True:
    params = {
        "page": page,
        "limit": limit
    }
    response = requests.get(base_url, params=params)
    response.raise_for_status()  # Raise an exception for HTTP errors
    data = response.json()

    products = data.get("products", [])

    if not products or len(products) < limit:
        # If no products or fewer than the limit, we've reached the last page
        for product in products:
            all_products_data.append({"id": product["id"], "handle": product["handle"]})
        break
    else:
        # Add all products from the current page
        for product in products:
            all_products_data.append({"id": product["id"], "handle": product["handle"]})

    page += 1

print(f"Found {len(all_products_data)} products.")
display(all_products_data)

Found 3824 products.


[{'id': 10162396299396, 'handle': 'pupa-wonder-me-shiny-bronzer'},
 {'id': 7203500097668, 'handle': 'pupa-wonder-me-blush'},
 {'id': 6590794039428, 'handle': 'pupa-wonder-cover-concealer'},
 {'id': 7203500064900, 'handle': 'pupa-vamp-stylo-liner'},
 {'id': 10162396168324, 'handle': 'pupa-vamp-eyeshadow-matt'},
 {'id': 10003214106756, 'handle': 'pupa-vamp-mascara-sexy-lashes'},
 {'id': 7203499901060,
  'handle': 'pupa-vamp-mascara-exceptional-volume-exagger'},
 {'id': 6583962730628, 'handle': 'pupa-vamp-mascara-all-in-one-101'},
 {'id': 10162397773956, 'handle': 'pupa-vamp-liquid-eyeshadow'},
 {'id': 8189990240388, 'handle': 'pupa-vamp-waterproof-2in1-lip-pencil'},
 {'id': 8189990109316, 'handle': 'pupa-vamp-waterproof-2in1-eye-pencil'},
 {'id': 10064654139524, 'handle': 'pupa-vamp-professional-liner-waterproof'},
 {'id': 7203499769988, 'handle': 'pupa-vamp-creamy-duo'},
 {'id': 7203499737220, 'handle': 'pupa-true-lips-pencil'},
 {'id': 10064654106756, 'handle': 'pupa-shock-plump-lip-gl

In [ ]:
len(all_products_data)

3824

In [ ]:
import requests
import json
import csv
from pathlib import Path
from typing import List, Dict

# ---------------- CONFIG ----------------
BASE_URL = "https://moustaphalabban.com/products/{}.js"
OUTPUT_JSON = "moustaphalabban_products.json"
OUTPUT_CSV = "moustaphalabban_products.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


# ---------------- FETCH FUNCTION ----------------
def fetch_product(handle: str) -> Dict:
    url = BASE_URL.format(handle)
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"[ERROR] Failed to fetch {handle}: {e}")
        return None


# ---------------- PARSE FUNCTION ----------------
def parse_product(product: Dict) -> List[Dict]:
    results = []

    # Product-level fields
    product_data = {
        "product_id": product.get("id"),
        "title": product.get("title"),
        "handle": product.get("handle"),
        "description": product.get("description"),
        "created_at": product.get("created_at"),
        "vendor": product.get("vendor"),
        "type": product.get("type"),
        "price": product.get("price"),
        "compare_at_price": product.get("compare_at_price"),
        "available": product.get("available"),
    }

    variants = product.get("variants", [])

    # If NO variants → still return one row
    if not variants:
        results.append(product_data)
        return results

    # Variant-level fields
    for variant in variants:
        row = product_data.copy()

        row.update({
            "variant_id": variant.get("id"),
            "variant_title": variant.get("title"),
            "variant_sku": variant.get("sku"),
            "variant_available": variant.get("available"),
            "variant_name": variant.get("name"),
            "variant_price": variant.get("price"),
            "variant_compare_at_price": variant.get("compare_at_price"),
            "variant_barcode": variant.get("barcode"),
            "variant_image": variant.get("featured_image", {}).get("src") if variant.get("featured_image") else None
        })

        results.append(row)

    return results


# ---------------- MAIN ----------------
def main():
    all_data = []

    for item in all_products_data:
        handle = item["handle"]
        print(f"[INFO] Fetching {handle}")

        product_json = fetch_product(handle)
        if not product_json:
            continue

        parsed_rows = parse_product(product_json)
        all_data.extend(parsed_rows)

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    # Save CSV
    if all_data:
        keys = all_data[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_data)

    print(f"[DONE] Saved {len(all_data)} rows")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

[INFO] Fetching pupa-wonder-me-shiny-bronzer
[INFO] Fetching pupa-wonder-me-blush
[INFO] Fetching pupa-wonder-cover-concealer
[INFO] Fetching pupa-vamp-stylo-liner
[INFO] Fetching pupa-vamp-eyeshadow-matt
[INFO] Fetching pupa-vamp-mascara-sexy-lashes
[INFO] Fetching pupa-vamp-mascara-exceptional-volume-exagger
[INFO] Fetching pupa-vamp-mascara-all-in-one-101
[INFO] Fetching pupa-vamp-liquid-eyeshadow
[INFO] Fetching pupa-vamp-waterproof-2in1-lip-pencil
[INFO] Fetching pupa-vamp-waterproof-2in1-eye-pencil
[INFO] Fetching pupa-vamp-professional-liner-waterproof
[INFO] Fetching pupa-vamp-creamy-duo
[INFO] Fetching pupa-true-lips-pencil
[INFO] Fetching pupa-shock-plump-lip-gloss
[INFO] Fetching pupa-shine-bright-all-in-one-palette
[INFO] Fetching pupa-fruit-lovers-body-scrub
[INFO] Fetching pupa-professionals-bb-cream-primer-nude
[INFO] Fetching pupa-professionals-bb-cream-primer-all-skin-types
[INFO] Fetching pupa-plump-care-eyebrow-gel
[INFO] Fetching pupa-pleasure-lip-oil
[INFO] Fetchin

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import csv

# ---------------- CONFIG ----------------
BASE_URL = "https://www.fahsbeauty.com/shop/page/{}/"
START_PAGE = 1   # you can change this
END_PAGE = 80     # increase if you want multiple pages

OUTPUT_JSON = "fahs_products.json"
OUTPUT_CSV = "fahs_products.csv"

HEADERS = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "accept-encoding": "gzip, deflate, br, zstd",
    "accept-language": "en-US,en;q=0.9,id;q=0.8,fa;q=0.7,ar;q=0.6,ms;q=0.5,ja;q=0.4,es;q=0.3",
    "cookie": "tk_ai=o0u360Q2F4oWFgf4L8YdAMaA",
    "priority": "u=0, i",
    "referer": "https://www.fahsbeauty.com/shop/",
    "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Microsoft Edge";v="146"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "document",
    "sec-fetch-mode": "navigate",
    "sec-fetch-site": "same-origin",
    "sec-fetch-user": "?1",
    "upgrade-insecure-requests": "1",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0"
}


# ---------------- FETCH ----------------
def fetch_page(page):
    url = BASE_URL.format(page)
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        res.raise_for_status()
        return res.text
    except Exception as e:
        print(f"[ERROR] Page {page}: {e}")
        return None


# ---------------- PARSE ----------------
def parse_products(html):
    soup = BeautifulSoup(html, "html.parser")
    results = []

    products = soup.select("div.c-product-grid__item")

    for product in products:
        # product title
        title_tag = product.select_one("h2.woocommerce-loop-product__title")

        # product link
        link_tag = product.select_one("a.woocommerce-LoopProduct-link")

        if not title_tag or not link_tag:
            continue

        name = title_tag.get_text(strip=True)
        url = link_tag.get("href")

        # Fix relative URLs
        if url.startswith("/"):
            url = "https://www.fahsbeauty.com" + url

        results.append({
            "product_name": name,
            "product_url": url
        })

    return results


# ---------------- MAIN ----------------
def main():
    all_products = []

    for page in range(START_PAGE, END_PAGE + 1):
        print(f"[INFO] Scraping page {page}")
        html = fetch_page(page)

        if not html:
            continue

        products = parse_products(html)
        all_products.extend(products)

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    # Save CSV
    if all_products:
        keys = all_products[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_products)

    print(f"[DONE] Total products: {len(all_products)}")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

[INFO] Scraping page 1
[ERROR] Page 1: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/1/
[INFO] Scraping page 2
[ERROR] Page 2: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/2/
[INFO] Scraping page 3
[ERROR] Page 3: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/3/
[INFO] Scraping page 4
[ERROR] Page 4: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/4/
[INFO] Scraping page 5
[ERROR] Page 5: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/5/
[INFO] Scraping page 6
[ERROR] Page 6: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/6/
[INFO] Scraping page 7
[ERROR] Page 7: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/7/
[INFO] Scraping page 8
[ERROR] Page 8: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/8/
[INFO] Scraping page 9
[ERROR] Page 9: 403 Client Error: Forbidden for u

In [3]:
!unzip -o /content/product_pages.zip -d /content/product_pages

Archive:  /content/product_pages.zip
   creating: /content/product_pages/product_pages/
  inflating: /content/product_pages/product_pages/4711-man-eau-de-toilette-400ml.html  
  inflating: /content/product_pages/product_pages/afnan-9-am-100ml-edp.html  
  inflating: /content/product_pages/product_pages/afnan-9-am-dive-homme-edp-100ml.html  
  inflating: /content/product_pages/product_pages/afnan-9-am-femme-edp-100ml.html  
  inflating: /content/product_pages/product_pages/afnan-9-pm-100ml-edp.html  
  inflating: /content/product_pages/product_pages/afnan-9-pm-elixir.html  
  inflating: /content/product_pages/product_pages/afnan-9-pm-femme-edp-100ml.html  
  inflating: /content/product_pages/product_pages/afnan-9pm-rebel-man-100ml.html  
  inflating: /content/product_pages/product_pages/afnan-cherry-bouquet-women-eau-de-perfume-80-ml.html  
  inflating: /content/product_pages/product_pages/afnan-concentrated-perfume-musk-abiyad-oil-20ml.html  
  inflating: /content/product_pages/product

In [ ]:
#@title FahsBeauty Parser V1

import json
import csv
from pathlib import Path
from bs4 import BeautifulSoup

# ---------------- CONFIG ----------------
INPUT_DIR = Path("product_pages/product_pages")
OUTPUT_JSON = "parsed_products.json"
OUTPUT_CSV = "parsed_products.csv"

BASE_URL = "https://www.fahsbeauty.com/shop/"


# ---------------- HELPERS ----------------
def get_text(el):
    return el.get_text(strip=True) if el else None


def get_all_text(el):
    return el.get_text(separator=" ", strip=True) if el else None


def extract_product(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    data = {}

    # ---------------- BASIC ----------------
    data["product_url"] = BASE_URL + file_path.stem + "/"

    # Title
    data["title"] = get_text(soup.select_one("h1.c-product__title"))

    # Short description
    data["short_description"] = get_all_text(
        soup.select_one("div.c-product__short-description")
    )

    # Full description
    data["description"] = get_all_text(
        soup.select_one("#tab-description")
    )

    # ---------------- BRAND ----------------
    brand = soup.select_one(".c-product__brands a")
    if not brand:
        brand = soup.select_one(".c-product__brand-logo-list a img")
        data["brand"] = brand.get("alt") if brand else None
    else:
        data["brand"] = get_text(brand)

    # ---------------- SKU ----------------
    data["sku"] = get_text(soup.select_one(".sku"))

    # ---------------- TAGS ----------------
    tags = soup.select(".tagged_as a")
    data["tags"] = [get_text(t) for t in tags] if tags else []

    # ---------------- CATEGORIES ----------------
    cats = soup.select(".posted_in a")
    data["categories"] = [get_text(c) for c in cats] if cats else []

    # ---------------- PRICE ----------------
    price = soup.select_one(".price .woocommerce-Price-amount")
    data["price"] = get_text(price)

    # ---------------- IMAGE ----------------
    img = soup.select_one(".c-product__slider-item a")
    data["image_url"] = img.get("href") if img else None

    return data


# ---------------- MAIN ----------------
def main():
    all_products = []

    files = list(INPUT_DIR.glob("*.html"))

    print(f"[INFO] Found {len(files)} files")

    for i, file in enumerate(files, 1):
        try:
            print(f"[{i}/{len(files)}] Parsing {file.name}")
            product = extract_product(file)
            all_products.append(product)

        except Exception as e:
            print(f"[ERROR] {file.name}: {e}")

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    # Save CSV
    if all_products:
        keys = all_products[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_products)

    print(f"\n[DONE] Parsed {len(all_products)} products")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

In [4]:
#@title FahsBeauty Parser V2

import json
import csv
from pathlib import Path
from bs4 import BeautifulSoup

# ---------------- CONFIG ----------------
INPUT_DIR = Path("product_pages/product_pages")
OUTPUT_JSON = "parsed_products.json"
OUTPUT_CSV = "parsed_products.csv"

BASE_URL = "https://www.fahsbeauty.com/shop/"


# ---------------- HELPERS ----------------
def get_text(el):
    return el.get_text(strip=True) if el else None


def get_all_text(el):
    return el.get_text(separator=" ", strip=True) if el else None


# ---------------- EXTRACT ----------------
def extract_product(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    data = {}

    # ---------------- BASIC ----------------
    data["product_url"] = BASE_URL + file_path.stem + "/"

    # Title
    data["title"] = get_text(soup.select_one("h1.c-product__title"))

    # Short description
    data["short_description"] = get_all_text(
        soup.select_one("div.c-product__short-description")
    )

    # Full description
    data["description"] = get_all_text(
        soup.select_one("#tab-description")
    )

    # ---------------- BRAND ----------------
    brand = soup.select_one(".c-product__brands a")
    if not brand:
        brand_img = soup.select_one(".c-product__brand-logo-list a img")
        data["brand"] = brand_img.get("alt") if brand_img else None
    else:
        data["brand"] = get_text(brand)

    # ---------------- SKU ----------------
    data["sku"] = get_text(soup.select_one(".sku"))

    # ---------------- TAGS ----------------
    tags = soup.select(".tagged_as a")
    data["tags"] = [get_text(t) for t in tags] if tags else []

    # ---------------- CATEGORIES ----------------
    cats = soup.select(".posted_in a")
    data["categories"] = [get_text(c) for c in cats] if cats else []

    # ---------------- STOCK ----------------
    stock_el = soup.select_one(".stock")
    if stock_el and "out-of-stock" in stock_el.get("class", []):
        data["stock"] = "out of stock"
    else:
        data["stock"] = "in stock"

    # ---------------- PRICE ----------------
    price_block = soup.select_one(".price")

    original_price = None
    current_price = None

    if price_block:
        # Original price (del)
        del_price = price_block.select_one("del .woocommerce-Price-amount")
        if del_price:
            original_price = get_text(del_price)

        # Current price (ins)
        ins_price = price_block.select_one("ins .woocommerce-Price-amount")
        if ins_price:
            current_price = get_text(ins_price)

        # If no discount → fallback
        if not current_price:
            normal_price = price_block.select_one(".woocommerce-Price-amount")
            current_price = get_text(normal_price)

    data["original_price"] = original_price
    data["current_price"] = current_price

    # ---------------- IMAGE ----------------
    img = soup.select_one(".c-product__slider-item a")
    data["image_url"] = img.get("href") if img else None

    return data


# ---------------- MAIN ----------------
def main():
    all_products = []

    files = list(INPUT_DIR.glob("*.html"))

    print(f"[INFO] Found {len(files)} files")

    for i, file in enumerate(files, 1):
        try:
            print(f"[{i}/{len(files)}] Parsing {file.name}")
            product = extract_product(file)
            all_products.append(product)

        except Exception as e:
            print(f"[ERROR] {file.name}: {e}")

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    # Save CSV
    if all_products:
        keys = all_products[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_products)

    print(f"\n[DONE] Parsed {len(all_products)} products")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

[INFO] Found 2394 files
[1/2394] Parsing clarins-skin-illusion-v-105n-30ml.html
[2/2394] Parsing afnan-turathi-femme-purple-women-eau-de-perfume-100-ml.html
[3/2394] Parsing dior-skin-forever-natural-nude-fdt-fl-2n.html
[4/2394] Parsing bourjois-velvet-the-pencil.html
[5/2394] Parsing prada-paradoxe-intense-eau-de-parfum.html
[6/2394] Parsing jean-paul-gaultier-divine.html
[7/2394] Parsing goldfield-banks-sunset-hour-eau-de-parfum.html
[8/2394] Parsing laverne-queen-rose-eau-de-parfum-200ml.html
[9/2394] Parsing guerlain-terracotta-mat-fluid-fdt-2c.html
[10/2394] Parsing issey-miyake-leau-dissey-pivoine-eau-de-toilette-intense.html
[11/2394] Parsing dunhill-icon-racing-red-man-100ml.html
[12/2394] Parsing narciso-rodriguez-all-of-me-intense.html
[13/2394] Parsing lacoste-touch-of-pink-pour-femme-eau-de-toilette.html
[14/2394] Parsing lacoste-rose-eau-fraiche-women-eau-de-toilette.html
[15/2394] Parsing armaf-odyssey-mega-limited-edition-vapo-100ml.html
[16/2394] Parsing max-factor-face

In [ ]:
import requests
import json
import csv
import time

# ---------------- CONFIG ----------------
URL = "https://www.lojaglamourosa.com/lb/en/task/latest/json/shop/ProductList/fetch"

START_PAGE = 1
END_PAGE = 42   # increase as needed
DELAY = 2

OUTPUT_JSON = "lojaglamourosa_products.json"
OUTPUT_CSV = "lojaglamourosa_products.csv"


# ---------------- HEADERS ----------------
# headers = {
#     "accept": "*/*",
#     "content-type": "application/x-www-form-urlencoded; charset=UTF-8",
#     "origin": "https://www.lojaglamourosa.com",
#     "referer": "https://www.lojaglamourosa.com/lb/en/skin-care",
#     "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/146.0.0.0 Safari/537.36",
#     "x-requested-with": "XMLHttpRequest",
#     "x-beevo-language": "1",
#     "x-beevo-language_slug": "en",
# }


# ---------------- BASE PAYLOAD ----------------
base_payload = {
    "filter_params[app_namespace]": "Shop",
    "filter_params[app_slug]": "shop",
    "filter_params[app_version]": "latest",
    "filter_params[widget_namespace]": "ProductList",
    "filter_params[widget_slug]": "product-list",
    "filter_params[widget_instance_id]": "26",
    "filter_params[category_id]": "5905fc97-8231-4f38-bae9-ac448ade2bf3",
    "filter_params[segment]": "skin-care",
    "filter_params[_url]": "/lb/en/api/latest/dot/shop/product-list/26",
    "filter_params[_beevo_decoded_params][0][category_id]": "5905fc97-8231-4f38-bae9-ac448ade2bf3",
    "filter_params[_beevo_decoded_params][0][segment]": "skin-care",
    "items_per_page": "100",
    "force_find_children": "0",
    "load_warehouses": "0"
}


# ---------------- FETCH ----------------
def fetch_page(page):
    payload = base_payload.copy()
    payload["page"] = str(page)

    try:
        res = requests.post(URL, headers=headers, data=payload, timeout=20)
        res.raise_for_status()
        return res.json()
    except Exception as e:
        print(f"[ERROR] Page {page}: {e}")
        return None


# ---------------- PARSE ----------------
def parse_products(data):
    results = []

    # ⚠️ structure may vary → adjust if needed
    products = data['actions'][0]['result']['products']

    for p in products:
        results.append({
            "product_id": p.get("id"),
            "name": p.get("name"),
            "slug": p.get("slug"),
            "brand": p.get("brand", {}).get("name") if p.get("brand") else None,
            "price": p.get("price"),
            "url": "https://www.lojaglamourosa.com" + p.get("url", ""),
            "image": p.get("image"),
        })

    return results


# ---------------- MAIN ----------------
def main():
    all_products = []

    for page in range(START_PAGE, END_PAGE + 1):
        print(f"[INFO] Fetching page {page}")

        data = fetch_page(page)
        if not data:
            continue

        products = parse_products(data)
        print(f"  → {len(products)} products")

        all_products.extend(products)

        time.sleep(DELAY)

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    # Save CSV
    if all_products:
        keys = all_products[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_products)

    print(f"\n[DONE] Total products: {len(all_products)}")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

[INFO] Fetching page 1
  → 100 products
[INFO] Fetching page 2
  → 100 products
[INFO] Fetching page 3
  → 100 products
[INFO] Fetching page 4
  → 100 products
[INFO] Fetching page 5
  → 100 products
[INFO] Fetching page 6
  → 100 products
[INFO] Fetching page 7


KeyboardInterrupt: 

In [ ]:
import requests

headers = {
    "accept": "*/*",
    "accept-encoding": "gzip, deflate, br, zstd",
    "accept-language": "en-US,en;q=0.9,id;q=0.8,fa;q=0.7,ar;q=0.6,ms;q=0.5,ja;q=0.4,es;q=0.3",
    "content-length": "667",
    "content-type": "application/x-www-form-urlencoded; charset=UTF-8",
    "cookie": "user_dataLayer=%7B%22logged%22%3A0%2C%22login_event_sent%22%3A0%7D; cconsent={'version':1,'categories':{'Performance':{'wanted':true},'Marketing':{'wanted':true},'Functional':{'wanted':true},'Mandatory':{'wanted':true}},'services':['doubleclick','Google Ads','Tidio','HubSpot','facebook','Beevo','google-tagmanager','Analytics']}; eg_params={'referrer':'https%3A%2F%2Fwww.lojaglamourosa.com%2Flb%2Fen'}; PHPSESSID=0a75f147efaf3823e95314bd8482f911; cf_clearance=zmXjDKCv5uuxTaHYgOn1byPWMrPytMlzoe1bKvhWN2k-1775638324-1.2.1.1-.WbK3VYLIIe6SEDoqPkCw5CivuauKlG3Mxeaf_RK86G6bqsxbNs51hMZcYIp1nS8WGVWv3rmozCiXS3gBjIu2mug5MhfpUDb2H1MITUFUUAWvtYPci2Ayi0SmZunaJrn5RCE0olb4MZJmpwop46gHdr3HPDtRdRBVVqbBgKoFSoc9KACx2WeA3IoT6cwQRsTYX4R9nWUEzjPncn3tvpiVojgx6yc9Yo5OCf6K6NUPUPFR5pMjYposS48bPJ8.RRRjMFcHgcy1vrYi3x2s3D5EeY5WNLMJmlmcf1eU5tfFrrKHAwc1EtoK5hUBIDB1oOvgcU1nrkeqvrMZA.UzF0IIw",
    "origin": "https://www.lojaglamourosa.com",
    "priority": "u=1, i",
    "referer": "https://www.lojaglamourosa.com/lb/en/skin-care?page=2",
    "sec-ch-ua": "\"Chromium\";v=\"146\", \"Not-A.Brand\";v=\"24\", \"Microsoft Edge\";v=\"146\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": "\"Windows\"",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0",
    "x-beevo-language": "1",
    "x-beevo-language_slug": "en",
    "x-beevo-layout": "3",
    "x-beevo-page": "144",
    "x-beevo-page_title": "Skin%20Care%20Lebanon%20Buy%20Online%20Loja%20Glamourosa",
    "x-requested-with": "XMLHttpRequest"
}

payload = {
    "filter_params[app_namespace]": "Shop",
    "filter_params[app_slug]": "shop",
    "filter_params[app_version]": "latest",
    "filter_params[widget_namespace]": "ProductList",
    "filter_params[widget_slug]": "product-list",
    "filter_params[widget_instance_id]": "26",
    # "filter_params[page]": "2",
    "filter_params[category_id]": "5905fc97-8231-4f38-bae9-ac448ade2bf3",
    "filter_params[segment]": "skin-care",
    "filter_params[_url]": "/lb/en/api/latest/dot/shop/product-list/26",
    "filter_params[_beevo_decoded_params][0][category_id]": "5905fc97-8231-4f38-bae9-ac448ade2bf3",
    "filter_params[_beevo_decoded_params][0][segment]": "skin-care",
    "items_per_page": "100",
    "force_find_children": "0",
    "load_warehouses": "0",
    "page": "3"
}

URL = "https://www.lojaglamourosa.com/lb/en/task/latest/json/shop/ProductList/fetch"

res = requests.post(URL, headers=headers, data=payload)
data = res.json()['actions'][0]['result']['products']
len(data)

100

In [ ]:
# hair-care 372c0dc3-970c-4958-8702-67addd50347a
# health-care 6c144456-3432-47d3-98b5-0bc090a1e4b1
# skin-care 5905fc97-8231-4f38-bae9-ac448ade2bf3
# makeup db98d4ed-e0fa-4e57-8c16-d503909826eb
# fragrances d6320798-3a94-4e98-b59e-9e08f6bb8f85
# man 409e5d07-76b8-4700-8f84-579f8c0a1bcf
# sun-care 1cea24ce-1a48-4fa6-984b-7c82f21b6e22

import requests
import json
import csv
import time
import re

# ---------------- CONFIG ----------------
URL = "https://www.lojaglamourosa.com/lb/en/task/latest/json/shop/ProductList/fetch"

DELAY = 1
OUTPUT_JSON = "skin-care_products_full.json"
OUTPUT_CSV = "skin-care_products_full.csv"


# ---------------- HEADERS ----------------
headers = {
    "accept": "*/*",
    "content-type": "application/x-www-form-urlencoded; charset=UTF-8",
    "origin": "https://www.lojaglamourosa.com",
    "referer": "https://www.lojaglamourosa.com/lb/en/skin-care",
    "user-agent": "Mozilla/5.0",
    "x-requested-with": "XMLHttpRequest",
    "x-beevo-language": "1",
    "x-beevo-language_slug": "en",
}


# ---------------- BASE PAYLOAD ----------------
base_payload = {
    "filter_params[app_namespace]": "Shop",
    "filter_params[app_slug]": "shop",
    "filter_params[app_version]": "latest",
    "filter_params[widget_namespace]": "ProductList",
    "filter_params[widget_slug]": "product-list",
    "filter_params[widget_instance_id]": "26",
    "filter_params[category_id]": "5905fc97-8231-4f38-bae9-ac448ade2bf3",
    "filter_params[segment]": "skin-care",
    "filter_params[_url]": "/lb/en/api/latest/dot/shop/product-list/26",
    "filter_params[_beevo_decoded_params][0][category_id]": "5905fc97-8231-4f38-bae9-ac448ade2bf3",
    "filter_params[_beevo_decoded_params][0][segment]": "skin-care",
    "items_per_page": "100",
    "force_find_children": "0",
    "load_warehouses": "0"
}


# ---------------- HELPERS ----------------
def extract_reference_from_image(url):
    """
    Extract code between 'shop-' and next '-'
    example: shop-rt-05774-xxx.jpg → rt-05774
    """
    if not url:
        return None
    match = re.search(r"shop-([a-z0-9\-]+?)-", url.lower())
    return match.group(1) if match else None


def extract_ean_from_segment(segment):
    """
    Extract last numeric part
    example: xxx-697045155767 → 697045155767
    """
    if not segment:
        return None
    match = re.search(r"(\d+)$", segment)
    return match.group(1) if match else None


# ---------------- FETCH ----------------
def fetch_page(page):
    payload = base_payload.copy()
    payload["page"] = str(page)

    try:
        res = requests.post(URL, headers=headers, data=payload, timeout=20)
        res.raise_for_status()
        return res.json()
    except Exception as e:
        print(f"[ERROR] Page {page}: {e}")
        return None


# ---------------- PARSE ----------------
def parse_products(data):
    results = []

    products = data.get("actions", [])[0].get("result", {}).get("products", [])

    for item in products:
        p = item.get("product", {})

        product_info = p.get("product", {})
        prices = p.get("prices", {})
        manufacturer = p.get("manufacturer", {})
        seo = p.get("seo_data", {})
        medias = p.get("medias", [])

        segment = seo.get("segment")

        # images
        image_urls = []
        for m in medias:
            if "shop-image-medium" in m:
                image_urls.append(m["shop-image-medium"])

        # reference number (from first image)
        # reference_number = extract_reference_from_image(
        #     image_urls[0] if image_urls else None
        # )

        results.append({
            "product_id": product_info.get("product_id"),
            "product_name": product_info.get("name"),
            "product_sku": product_info.get("sku"),
            "description": product_info.get("description"),
            "stock": product_info.get("stock"),

            "category_name": p.get("category", {}).get("category_name"),
            "manufacturer": manufacturer.get("name"),

            "base_price_with_tax_unformatted": prices.get("base_price_with_tax_unformatted"),
            "discount_amount_with_tax_unformatted": prices.get("discount_amount_with_tax_unformatted"),
            "discount_percentage": prices.get("discount_percentage"),
            "sales_price_with_tax_unformatted": prices.get("sales_price_with_tax_unformatted"),

            "images": image_urls,

            "reference_number": product_info.get("sku"),
            "ean_number": extract_ean_from_segment(segment),

            "product_url": f"https://www.lojaglamourosa.com/lb/en/{segment}"
        })

    return results


# ---------------- MAIN ----------------
def main():
    all_products = []
    page = 1

    while True:
        print(f"[INFO] Fetching page {page}")

        data = fetch_page(page)
        if not data:
            break

        products = parse_products(data)
        print(f"  → {len(products)} products")

        if not products:
            break

        all_products.extend(products)

        # STOP condition
        if len(products) < 100:
            print("[INFO] Last page reached")
            break

        page += 1
        time.sleep(DELAY)

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    # Save CSV
    if all_products:
        keys = all_products[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_products)

    print(f"\n[DONE] Total products: {len(all_products)}")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

[INFO] Fetching page 1
  → 100 products
[INFO] Fetching page 2
  → 100 products
[INFO] Fetching page 3
  → 100 products
[INFO] Fetching page 4
  → 100 products
[INFO] Fetching page 5
  → 100 products
[INFO] Fetching page 6
  → 100 products
[INFO] Fetching page 7
  → 100 products
[INFO] Fetching page 8
  → 100 products
[INFO] Fetching page 9
  → 100 products
[INFO] Fetching page 10
  → 100 products
[INFO] Fetching page 11
  → 100 products
[INFO] Fetching page 12
  → 100 products
[INFO] Fetching page 13
  → 100 products
[INFO] Fetching page 14
  → 100 products
[INFO] Fetching page 15
  → 100 products
[INFO] Fetching page 16
  → 100 products
[INFO] Fetching page 17
  → 100 products
[INFO] Fetching page 18
  → 100 products
[INFO] Fetching page 19
  → 100 products
[INFO] Fetching page 20
  → 100 products
[INFO] Fetching page 21
  → 100 products
[INFO] Fetching page 22
  → 100 products
[INFO] Fetching page 23
  → 100 products
[INFO] Fetching page 24
  → 100 products
[INFO] Fetching page 25
 

In [ ]:
all_products[0]

NameError: name 'all_products' is not defined

In [ ]:
import pandas as pd
import json
from pathlib import Path

# --- Combine CSVs ---
combined_csv_data = []
for file_path in Path('/content').glob('*_products_full.csv'):
    try:
        df = pd.read_csv(file_path)
        combined_csv_data.append(df)
        print(f"[INFO] Loaded CSV: {file_path.name}")
    except Exception as e:
        print(f"[ERROR] Failed to load CSV {file_path.name}: {e}")

if combined_csv_data:
    all_csv_products_df = pd.concat(combined_csv_data, ignore_index=True)
    all_csv_products_df.to_csv('all_products.csv', index=False)
    print(f"[DONE] Combined {len(combined_csv_data)} CSV files into 'all_products.csv' with {len(all_csv_products_df)} rows.")
else:
    print("[WARN] No CSV files found to combine.")

# --- Combine JSONs ---
combined_json_data = []
for file_path in Path('/content').glob('*_products_full.json'):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                combined_json_data.extend(data)
            else:
                print(f"[WARN] JSON file {file_path.name} does not contain a list of products. Skipping.")
        print(f"[INFO] Loaded JSON: {file_path.name}")
    except Exception as e:
        print(f"[ERROR] Failed to load JSON {file_path.name}: {e}")

if combined_json_data:
    with open('all_products.json', 'w', encoding='utf-8') as f:
        json.dump(combined_json_data, f, indent=2, ensure_ascii=False)
    print(f"[DONE] Combined {len(combined_json_data)} products from JSON files into 'all_products.json'.")
else:
    print("[WARN] No JSON files found to combine.")

[INFO] Loaded CSV: hair-care_products_full.csv
[INFO] Loaded CSV: bath_body_products_full.csv
[INFO] Loaded CSV: skin-care_products_full.csv
[INFO] Loaded CSV: fragrances_products_full.csv
[INFO] Loaded CSV: sun-care_products_full.csv
[INFO] Loaded CSV: man_products_full.csv
[INFO] Loaded CSV: health-care_products_full.csv
[INFO] Loaded CSV: makeup_products_full.csv
[DONE] Combined 8 CSV files into 'all_products.csv' with 13635 rows.
[INFO] Loaded JSON: hair-care_products_full.json
[INFO] Loaded JSON: sun-care_products_full.json
[INFO] Loaded JSON: fragrances_products_full.json
[INFO] Loaded JSON: health-care_products_full.json
[INFO] Loaded JSON: man_products_full.json
[INFO] Loaded JSON: skin-care_products_full.json
[INFO] Loaded JSON: makeup_products_full.json
[INFO] Loaded JSON: bath_body_products_full.json
[DONE] Combined 13635 products from JSON files into 'all_products.json'.
